# Strict Gemini Retraining (No Default Labels)

This notebook does:
1. Clone repo
2. Upload your new images
3. Label with Gemini only (**no fallback depth**)
4. Fine-tune from `models/best_flood_model_water_aware.pth`
5. Download trained model


In [ ]:
!pip install -q torch torchvision pillow tqdm google-genai
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
%cd /content
!rm -rf flood-depth-estimator
!git clone https://github.com/Mishra-Kaumod/flood-depth-estimator.git
%cd /content/flood-depth-estimator
!ls

In [ ]:
from pathlib import Path
import csv
import re
import time
import shutil
import getpass
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, random_split
from PIL import Image
from tqdm import tqdm
from google.colab import files

REPO_DIR = Path('/content/flood-depth-estimator')
BASE_MODEL = REPO_DIR / 'models' / 'best_flood_model_water_aware.pth'
WORK_DIR = REPO_DIR / 'colab_strict_run'
IMAGES_DIR = WORK_DIR / 'images'
LABELS_CSV = WORK_DIR / 'labels.csv'
VERSION = 'v4'
EPOCHS = 12
BATCH_SIZE = 8
LR = 1e-4
REQUEST_DELAY = 3.0
RETRY_ATTEMPTS = 5
RETRY_BACKOFF = 4.0
MODEL_NAME = 'gemini-1.5-flash'

WORK_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

if not BASE_MODEL.exists():
    raise FileNotFoundError(f'Base model missing: {BASE_MODEL}')

print('Upload your NEW training images (jpg/png/webp)...')
uploaded = files.upload()
if len(uploaded) < 20:
    raise RuntimeError(f'Need at least 20 images, got {len(uploaded)}')

for name, data in uploaded.items():
    with open(IMAGES_DIR / name, 'wb') as f:
        f.write(data)

image_paths = sorted([p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}])
print('Images ready:', len(image_paths))

api_key = getpass.getpass('Gemini API key: ').strip()
if not api_key:
    raise RuntimeError('Gemini API key is required')

def make_gemini_callable(key: str, model_name: str):
    try:
        from google import genai as modern_genai
        from google.genai import types as modern_types
        client = modern_genai.Client(api_key=key)
        def call(image_path: Path, prompt: str) -> str:
            with open(image_path, 'rb') as f:
                payload = f.read()
            mime = 'image/jpeg' if image_path.suffix.lower() in {'.jpg','.jpeg'} else 'image/png'
            resp = client.models.generate_content(
                model=model_name,
                contents=[prompt, modern_types.Part.from_bytes(data=payload, mime_type=mime)],
            )
            text = getattr(resp, 'text', None)
            if not text:
                raise RuntimeError('Empty Gemini response')
            return text
        return call
    except Exception:
        import google.generativeai as legacy_genai
        legacy_genai.configure(api_key=key)
        gm = legacy_genai.GenerativeModel(model_name)
        def call(image_path: Path, prompt: str) -> str:
            img = Image.open(image_path).convert('RGB')
            resp = gm.generate_content([prompt, img])
            text = getattr(resp, 'text', None)
            if not text:
                raise RuntimeError('Empty Gemini response')
            return text
        return call

def parse_depth_strict(text: str) -> float:
    m = re.search(r'(-?\d+(\.\d+)?)', text)
    if not m:
        raise RuntimeError(f'No numeric value in Gemini response: {text[:100]}')
    value = float(m.group(1))
    if not (0.0 <= value <= 150.0):
        raise RuntimeError(f'Depth out of range: {value}')
    return value

gemini_call = make_gemini_callable(api_key, MODEL_NAME)
prompt = (
    'Estimate flood WATER DEPTH in centimeters from this image. '
    'Use visible cues (person/car/bike/building/curb waterline). '
    'Return ONLY one numeric value in cm.'
)

rows = []
for p in tqdm(image_paths, desc='Gemini strict labeling'):
    depth = None
    last_err = None
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            text = gemini_call(p, prompt)
            depth = parse_depth_strict(text)
            break
        except Exception as e:
            last_err = e
            print(f'[retry {attempt}/{RETRY_ATTEMPTS}] {p.name}: {e}')
            time.sleep(RETRY_BACKOFF * attempt)
    if depth is None:
        raise RuntimeError(f'Gemini labeling failed for {p.name}. Last error: {last_err}')
    rows.append({'filename': p.name, 'depth_cm': f'{depth:.2f}'})
    time.sleep(REQUEST_DELAY)

with open(LABELS_CSV, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['filename','depth_cm'])
    w.writeheader()
    w.writerows(rows)
print('Labels saved:', LABELS_CSV)

def build_model():
    m = models.efficientnet_b0(weights=None)
    feat = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(feat, 256), nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(256, 128), nn.ReLU(),
        nn.Linear(128, 1), nn.Sigmoid(),
    )
    return m

def load_base(m):
    ck = torch.load(BASE_MODEL, map_location=DEVICE, weights_only=False)
    state = ck.get('model_state_dict', ck) if isinstance(ck, dict) else ck
    target = m.state_dict()
    matched = {k:v for k,v in state.items() if k in target and target[k].shape == v.shape}
    if not matched:
        remap = {}
        for k,v in state.items():
            nk = None
            if k.startswith('backbone.'):
                nk = 'features.' + k[len('backbone.'):]
            elif k.startswith('head.0.'):
                nk = k.replace('head.0.','classifier.1.')
            elif k.startswith('head.4.'):
                nk = k.replace('head.4.','classifier.4.')
            elif k.startswith('head.8.'):
                nk = k.replace('head.8.','classifier.6.')
            if nk and nk in target and target[nk].shape == v.shape:
                remap[nk] = v
        matched = remap
    if len(matched) < 50:
        raise RuntimeError(f'Base model load failed; matched tensors: {len(matched)}')
    target.update(matched)
    m.load_state_dict(target)
    print(f'Loaded base tensors: {len(matched)}/{len(target)}')

class DepthDS(Dataset):
    def __init__(self, image_dir: Path, labels_csv: Path):
        self.items = []
        with open(labels_csv, 'r', encoding='utf-8') as f:
            for r in csv.DictReader(f):
                p = image_dir / r['filename']
                if p.exists():
                    self.items.append((p, float(r['depth_cm'])))
        if not self.items:
            raise RuntimeError('No labeled items found')
        self.tf = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, d = self.items[i]
        x = self.tf(Image.open(p).convert('RGB'))
        y = torch.tensor(d/100.0, dtype=torch.float32)
        return {'image': x, 'target': y}

model = build_model().to(DEVICE)
load_base(model)

dataset = DepthDS(IMAGES_DIR, LABELS_CSV)
n_val = max(1, int(0.2 * len(dataset)))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

for p in model.features.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True

optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-4)
criterion = nn.SmoothL1Loss(beta=0.05)
best = {'val_loss': float('inf')}

for epoch in range(EPOCHS):
    model.train()
    t_loss = 0.0
    for b in train_loader:
        x = b['image'].to(DEVICE)
        y = b['target'].to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        p = model(x)
        loss = criterion(p, y)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
    t_loss /= max(1, len(train_loader))

    model.eval()
    v_loss = 0.0
    v_mae = 0.0
    with torch.no_grad():
        for b in val_loader:
            x = b['image'].to(DEVICE)
            y = b['target'].to(DEVICE).unsqueeze(1)
            p = model(x)
            v_loss += criterion(p, y).item()
            v_mae += torch.abs((p - y) * 100.0).mean().item()
    v_loss /= max(1, len(val_loader))
    v_mae /= max(1, len(val_loader))
    print(f'Epoch {epoch+1}/{EPOCHS} | train_loss={t_loss:.5f} | val_loss={v_loss:.5f} | val_mae={v_mae:.2f}cm')
    if v_loss < best['val_loss']:
        best = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'val_loss': v_loss,
            'val_mae': v_mae,
            'training_method': 'strict_gemini_notebook'
        }

out_version = REPO_DIR / 'models' / f'best_flood_model_{VERSION}.pth'
out_default = REPO_DIR / 'models' / 'best_flood_model_water_aware.pth'
torch.save(best, out_version)
shutil.copy2(out_version, out_default)
print('Saved:', out_version)
print('Updated default:', out_default)
files.download(str(out_version))